# 🏸 Badminton Action Classification — BiLSTM + MediaPipe

**Based on**: *Badminton Action Classification Based on Human Skeleton Data Extracted by AlphaPose* (ICSMD 2023)

**Improvements**:
- AlphaPose → **MediaPipe Pose** (lighter, pip-installable)
- LSTM → **Bidirectional LSTM + Attention** (better temporal modelling)
- Targeting **85%+** accuracy vs paper's 80%

---
**Runtime**: Change to `Runtime → Change runtime type → T4 GPU` for faster training.

## 1. Setup Environment

In [ ]:
# Install all dependencies
!pip install -q mediapipe opencv-python-headless scikit-learn seaborn tqdm
print('✅ Dependencies installed')

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU detected. Training will be slower.')
    print('   Go to Runtime → Change runtime type → T4 GPU')

## 2. Download Dataset from Kaggle

In [ ]:
# Option A: Upload your kaggle.json
from google.colab import files
import os

os.makedirs('/root/.kaggle', exist_ok=True)
print('Upload your kaggle.json API key (from https://www.kaggle.com/settings)')
uploaded = files.upload()

# Move to ~/.kaggle/
for fn in uploaded.keys():
    os.rename(fn, f'/root/.kaggle/{fn}')
!chmod 600 /root/.kaggle/kaggle.json
print('✅ kaggle.json configured')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d imrankhan75/badminton-action-classification -p /content/raw_data

import zipfile, os
zip_path = '/content/raw_data/badminton-action-classification.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/raw_data/')

# List what was extracted
for root, dirs, files_list in os.walk('/content/raw_data'):
    level = root.replace('/content/raw_data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:3]:
            print(f'{subindent}{f}')

In [ ]:
import os, shutil

# Set this to the folder containing your 4 action subdirectories
# Adjust if the Kaggle zip structure differs
RAW_DIR = '/content/raw_data'
DATA_DIR = '/content/data'

ACTION_MAP = {
    'backhand drive':    'backhand_drive',
    'backhand_drive':    'backhand_drive',
    'backhand net shot': 'backhand_net_shot',
    'backhand_net_shot': 'backhand_net_shot',
    'forehand clear':    'forehand_clear',
    'forehand_clear':    'forehand_clear',
    'forehand drive':    'forehand_drive',
    'forehand_drive':    'forehand_drive',
}

os.makedirs(DATA_DIR, exist_ok=True)

def find_and_move(src_root):
    for item in os.listdir(src_root):
        src = os.path.join(src_root, item)
        if os.path.isdir(src) and item.lower().replace(' ', '_') in ACTION_MAP.values():
            canonical = item.lower().replace(' ', '_')
            dst = os.path.join(DATA_DIR, canonical)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'  {item} → {canonical} ({len(os.listdir(dst))} files)')

find_and_move(RAW_DIR)
for sub in os.listdir(RAW_DIR):
    sub_path = os.path.join(RAW_DIR, sub)
    if os.path.isdir(sub_path):
        find_and_move(sub_path)

print('\n✅ Data organised in', DATA_DIR)

## 3. Clone / Mount Project Code

In [ ]:
# Option A: If you pushed to GitHub
# !git clone https://github.com/YOUR_USERNAME/badminton-action-classification.git /content/project

# Option B: Write project files inline (paste your config.py etc.)
# For Colab convenience, we'll write the key files directly:

import os
os.makedirs('/content/project/model', exist_ok=True)
os.makedirs('/content/project/keypoint_extraction', exist_ok=True)
os.makedirs('/content/project/utils', exist_ok=True)
os.makedirs('/content/project/results', exist_ok=True)
os.makedirs('/content/project/model/saved_models', exist_ok=True)

import sys
sys.path.insert(0, '/content/project')
print('✅ Project directories created')

In [ ]:
%%writefile /content/project/config.py
import os

BASE_DIR = '/content/project'
DATA_DIR = '/content/data'
KEYPOINTS_DIR = '/content/keypoints'
MODEL_SAVE_DIR = '/content/project/model/saved_models'
RESULTS_DIR = '/content/project/results'

ACTION_CLASSES = ['backhand_drive', 'backhand_net_shot', 'forehand_clear', 'forehand_drive']
NUM_CLASSES = 4
FRAMES_PER_VIDEO = 10
NUM_LANDMARKS = 33
FEATURES_PER_FRAME = 66

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42

INPUT_SIZE   = 66
HIDDEN_SIZE  = 128
NUM_LAYERS   = 3
DROPOUT      = 0.3
FC_HIDDEN    = 64

BATCH_SIZE     = 32
NUM_EPOCHS     = 150
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4
LR_PATIENCE    = 15
LR_FACTOR      = 0.5
EARLY_STOP_PAT = 25

NUM_WORKERS = 2
PIN_MEMORY  = True

## 4. Extract Skeleton Keypoints (MediaPipe)

In [ ]:
# Copy extractor from your uploaded files, or run inline:
import sys, os
sys.path.insert(0, '/content/project')

# If you uploaded the project files, run extraction:
# from keypoint_extraction.mediapipe_extractor import run_extraction
# run_extraction(data_dir='/content/data', keypoints_dir='/content/keypoints')

# ── Inline extraction (no file upload needed) ─────────────────────────────
import mediapipe as mp
import cv2
import numpy as np
from tqdm.notebook import tqdm

mp_pose = mp.solutions.pose
DATA_DIR = '/content/data'
KEYPOINTS_DIR = '/content/keypoints'
ACTION_CLASSES = ['backhand_drive', 'backhand_net_shot', 'forehand_clear', 'forehand_drive']
FRAMES_PER_VIDEO = 10

def extract_video(video_path, n_frames=10):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    indices = np.linspace(0, total-1, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    
    seq = np.zeros((n_frames, 66), dtype=np.float32)
    with mp_pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.5) as pose:
        for i, frame in enumerate(frames):
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = pose.process(rgb)
            if result.pose_landmarks:
                coords = []
                for lm in result.pose_landmarks.landmark:
                    coords.extend([lm.x, lm.y])
                seq[i] = coords
    return seq

os.makedirs(KEYPOINTS_DIR, exist_ok=True)
video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
saved, failed = 0, 0

for action in ACTION_CLASSES:
    src = os.path.join(DATA_DIR, action)
    dst = os.path.join(KEYPOINTS_DIR, action)
    os.makedirs(dst, exist_ok=True)
    if not os.path.isdir(src):
        print(f'⚠️  Missing: {src}')
        continue
    vids = [f for f in os.listdir(src) if os.path.splitext(f)[1].lower() in video_exts]
    print(f'[{action}] {len(vids)} videos')
    for vf in tqdm(vids, desc=action):
        out = os.path.join(dst, os.path.splitext(vf)[0] + '.npy')
        if os.path.exists(out):
            saved += 1
            continue
        kp = extract_video(os.path.join(src, vf))
        if kp is not None:
            np.save(out, kp)
            saved += 1
        else:
            failed += 1

print(f'\n✅ Extraction done. Saved: {saved} | Failed: {failed}')

## 5. Train the BiLSTM Model

In [ ]:
# Upload your project Python files to /content/project/ first, then:
import sys
sys.path.insert(0, '/content/project')

from model.train import train
from utils.visualize import plot_training_curves

history = train(num_epochs=150, checkpoint_name='best_bilstm.pth')

# Plot training curves
from IPython.display import Image
plot_training_curves(
    history['train_loss'], history['val_loss'],
    history['train_acc'],  history['val_acc'],
    save_path='/content/project/results/training_curves.png'
)
Image('/content/project/results/training_curves.png')

## 6. Evaluate on Test Set

In [ ]:
from model.evaluate import evaluate
from IPython.display import Image, display

metrics = evaluate(checkpoint_name='best_bilstm.pth')

print(f'\n🏆 Overall Test Accuracy : {metrics["overall_accuracy"]:.2%}')
print(f'   Paper LSTM Baseline   : 80.00%')
print('\nPer-class accuracy:')
for cls, acc in metrics['per_class_accuracy'].items():
    bar = '█' * int(acc * 20)
    print(f'  {cls:<25} {bar:<20} {acc:.1%}')

display(Image('/content/project/results/confusion_matrix.png'))
display(Image('/content/project/results/per_class_accuracy.png'))

## 7. Inference on a Single Video

In [ ]:
import torch
import numpy as np
from model.evaluate import load_best_model

ACTION_CLASSES = ['backhand_drive', 'backhand_net_shot', 'forehand_clear', 'forehand_drive']

def predict_video(video_path: str, checkpoint_name: str = 'best_bilstm.pth') -> dict:
    """Predict the action in a single video file."""
    # Extract keypoints
    kp = extract_video(video_path)  # defined in cell 4
    if kp is None:
        return {'error': 'Could not extract keypoints'}
    
    # Load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, _ = load_best_model(checkpoint_name, device)
    
    # Inference
    x = torch.from_numpy(kp).unsqueeze(0).to(device)   # (1, 10, 66)
    with torch.no_grad():
        logits, attn_weights = model(x, return_attention=True)
        probs = torch.softmax(logits, dim=1).squeeze(0)
    
    pred_idx = probs.argmax().item()
    return {
        'prediction': ACTION_CLASSES[pred_idx],
        'confidence': f'{probs[pred_idx].item():.1%}',
        'all_probs': {c: f'{p.item():.1%}' for c, p in zip(ACTION_CLASSES, probs)},
        'attention': attn_weights.squeeze().cpu().tolist(),
    }

# Usage:
# result = predict_video('/path/to/your/video.mp4')
# print(result)
print('✅ predict_video() is ready. Pass any video path to classify it.')